# Gain attribution with floor check

Leak-free predictive training with a shared encoder and SIGReg. **Rule:** the encoder may simplify the target
latent only in directions where the predictor has stopped improving *and* sits near its achievable error floor.
Spec: `docs/gain_attribution_one_pager.pdf`. Code: `gaf/`. Background and design decisions: `README.md`.

**Stages**
1. Data and premise: what the clips look like, whether slow content alone can satisfy SIGReg, how hard each group is to learn.
2. Baselines (`lewm`, `sg`, `ema`): do they keep slow content and drop the rest?
3. Floors (`detached` + floor probes on the baseline encoders): which latent directions sit on an irreducible noise floor?
4. Fixes (`target`, `context`, `both`): keep fast and hard content, still drop noise?
5. Diagnostics: S over time, leak before and after projection.
6. Optional ablations.

Every run is resumable: finished runs in `OUT` are skipped, so after an interruption just re-run the cell.

In [ ]:
%pip install -q -e .
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
from IPython.display import display
import pandas as pd
from gaf.env import EnvCfg, GROUPS
from gaf.train import Cfg, ARMS, premise, run_gt
from gaf import analysis as A

OUT = "/workspace/results"   # on the RunPod network volume, survives pod restarts
SEEDS = (0, 1, 2)

# ---- the knobs ---------------------------------------------------------------------------
ENV = EnvCfg(
    res=32, T=3,                 # 32x32 RGB, 2 context frames + 1 target
    n_slow=16, slow_rho=0.98,    # slow patterns: Gaussian across clips, OU within a clip
    n_noise=8,                   # noise patterns: fresh every frame
    v_act=0.3,                   # fast sprite: p' = p + v_act * a
    hard_k=2, h_act=0.4,         # hard sprite: q' = k q + h_act * a (mod 1)
    a_slow=1.0, a_noise=1.0, a_fast=4.0, a_hard=4.0, pixel_noise=0.05,  # sprite peak vs pattern RMS
)
COMMON = dict(env=ENV, D=16, steps=20_000, batch=256, lr=1e-3, lam=0.1, wd=0.01,
              basis="resid", floor_steps=1500, floor_warm=True)
pd.set_option("display.precision", 3)

## 1 Data and premise
Four kinds of content per clip (rows = clips, columns = frames):
**slow** smooth colour patterns (OU, ρ≈1); **fast** white sprite pushed by the action; **hard** red sprite on an
action-driven chaotic circle map; **noise** patterns of the same kind as slow but fresh every frame.
Actions a_t ∈ [−1, 1]² (random policy) act between frames; the predictor sees them, the encoder does not.

In [ ]:
A.show_clips(ENV, n=4);

**Premise.** The failure needs slow content that can fill SIGReg's budget on its own. Below: SIGReg of hand-made
latents of size D built from one group each (whitened, zero-padded if the group has fewer than D factors).
`slow` close to `gaussian` = the encoder can satisfy SIGReg with slow content alone, so nothing forces it to keep the rest.

In [ ]:
display(premise(ENV, D=COMMON["D"]).to_frame().T)

**Learnability on ground truth** (no encoder): a predictor from the true context factors and actions to the true
target factors. Slow should be learned almost instantly, fast quickly, hard more slowly, noise never (R² ≈ 0).
The second table hides the actions: fast and hard must collapse to their caps (≈ 0.25, ≈ 0.05), otherwise they are
not really action-conditioned.

**Floor budget check.** The floor probe must learn what the predictor can learn, or it overestimates L* and opens the gate
on still-learnable content (the probe-bias caveat). Compare the steps the ground-truth predictor needs for 90 % of
its final R² on `hard` with the probe budget `floor_steps` (× refreshes kept, since the probe is warm-started).

In [ ]:
gt = run_gt(ENV, steps=5000)
gt_noact = run_gt(ENV, steps=5000, actions=False)
ax = gt.plot(figsize=(6, 3.4), color=[A.GROUP_COLOR[g] for g in GROUPS])
ax.set(xlabel="step", ylabel="held-out R²", ylim=(-0.05, 1.02)); ax.legend(loc="lower right")
display(pd.concat({"with actions": gt.iloc[[round(i * (len(gt) - 1) / 4) for i in range(5)]],
                   "without actions": gt_noact.iloc[[-1]]}))
t90 = {g: (int(gt.index[(gt[g] >= 0.9 * gt[g].iloc[-1]).values.argmax()]) if gt[g].iloc[-1] > 0.5
           else None) for g in GROUPS[:3]}   # None: not learned within run_gt's budget
print("steps to 90% of final R² on ground truth:", t90)
if t90["hard"] is None or t90["hard"] > COMMON["floor_steps"]:
    print(f"WARNING: hard needs {t90['hard'] or 'more than 5000'} steps on ground truth, the probe gets "
          f"{COMMON['floor_steps']} per refresh (warm start accumulates them). If hard directions get high "
          "floors and leave S while the predictor is still learning them, raise floor_steps.")

## 2 Baselines
`lewm`: full gradient through both paths. `sg`: stop-grad target (the leak is unchecked). `ema`: EMA target encoder.
Expected: slow content kept, fast and hard content dropped (or never learned), noise dropped.
The first run prints its speed: multiply by the number of runs to plan the session.

In [ ]:
A.execute(A.grid(A.BASELINES, SEEDS, **COMMON), OUT)
df = A.collect(OUT)
t_base = A.table(df[df.arm.isin(A.BASELINES) & (df.tag == "")])
display(t_base)
display(A.baseline_verdict(t_base))
A.bars(t_base); A.curves(df, OUT, A.BASELINES);

## 3 Floors
**(a)** `detached`: the encoder sees only SIGReg; the predictor trains on fully detached latents. Its L* history is the
noise floor of a latent that prediction never shaped.
**(b)** Floor probes on the final encoders of the baselines: one fresh predictor per frozen encoder, per-direction
held-out error L*_j, plotted against how much of direction j is noise. Noise directions should sit high, slow ones near 0.

In [ ]:
A.execute(A.grid(["detached"], SEEDS, **COMMON), OUT)
df = A.collect(OUT)
display(A.table(df[df.arm == "detached"]))
cfgs = [c for c in A.grid(["detached", "sg", "lewm"], SEEDS[:1], **COMMON)]
fl = pd.concat(A.floors_of(c, OUT) for c in cfgs)
display(fl.groupby("arm")[["floor", *GROUPS]].mean())
A.floor_scatter(fl);

## 4 Fixes
`target`: open target path on saturated directions only. `context`: leak projection on the context gradient only.
`both`: the full method. Success (vs. `sg`): fast and hard R² up by ≥ 0.3, noise within 0.1, slow still ≥ 0.5
(with n_slow = D, slow alone fills the latent, so keeping the sprites must cost some slow R²).

In [ ]:
A.execute(A.grid(A.FIXES, SEEDS, **COMMON), OUT)
df = A.collect(OUT)
t_all = A.table(df[df.tag == ""])
display(t_all)
display(A.fix_verdict(t_all, ref="sg"))
A.bars(t_all); A.curves(df, OUT, ["sg", *A.FIXES], tag="");

## 5 Diagnostics
- `n_S` protected directions; `leak_pos` share of diagnostic steps with ℓ > 0; `leak_cos` / `leak_cos_post` cosine of
  g_tgt,S and the applied context-side AdamW update before / after projection (≈ 0 after); `proj_ratio` norm ratio.
- `lreal_S`, `lreal_open`: exact check ℓreal = L(θ′, z_t(θ)) − L(θ′, z_t(θ′)) summed over S / over the open directions.
  After projection, `lreal_S` should be ≈ 0 (SIGReg and Adam momentum still move the target a little).
- `wb_ratio` within/between-clip variance of z (slow collapse → small), `rank_dz` effective rank of z_t − z_{t−1}.

In [ ]:
display(A.diag_table(df[df.tag == ""]))
A.leak_panel(df, OUT, ["sg", *A.FIXES], tag="");

Per-direction view of one run: which directions are protected, their relative gap and excess over the floor, and
which group each direction carries. In a working run, fast/hard directions stay in S while they are being learned and
noise directions leave S.

In [ ]:
rid = A.select(df, arm="both", seed=0, tag="").run_id.iloc[0]
A.gate_map(OUT, rid);
rid = A.select(df, arm="sg", seed=0, tag="").run_id.iloc[0]
A.gate_map(OUT, rid);

## 6 Ablations (optional)
Each changes one knob of `both`; `tag` keeps them apart in the tables. Uncomment what you need.

In [ ]:
ABL = {
    "proj=grad": dict(proj_on="grad"),      # one-pager sketch: project the raw gradient only
    "open-unprojected": dict(proj_open=False),  # open target path not projected against S
    "basis=dim": dict(basis="dim"),         # gate per latent dim (arbitrary under SIGReg)
    # "alpha=0.5": dict(alpha=0.5),         # partial projection (over-projection caveat)
    # "cold-floor": dict(floor_warm=False), # fresh probe at every refresh
    # "wd=0": dict(wd=0.0),
    # "soft": dict(soft=True),              # soft open-path weights
    # "sgd": dict(opt="sgd", lr=0.05),      # plain SGD: the derivation holds exactly
}
cfgs = [c for name, kw in ABL.items() for c in A.grid(["both"], SEEDS, **{**COMMON, **kw, "tag": name})]
A.execute(cfgs, OUT)
df = A.collect(OUT)
display(A.select(df, arm="both").groupby("tag")[A.R2 + ["n_S", "leak_cos_post", "lreal_S"]].mean())

## Environment sweeps (optional)
The failure and the fix depend on how salient and how hard the fast content is. Example: harder chaotic map and a
fainter fast sprite. Results go to a separate folder so tables stay comparable.

In [ ]:
# ENV2 = EnvCfg(**{**ENV.__dict__, "hard_k": 3, "a_fast": 2.0})   # harder map, fainter sprite
# ENV2 = EnvCfg(**{**ENV.__dict__, "n_slow": 12})                   # spare latent room (D=16)
# OUT2 = OUT + "/hard3"
# A.execute(A.grid(["sg", "both"], SEEDS, **{**COMMON, "env": ENV2}), OUT2)
# display(A.table(A.collect(OUT2)))

## All runs

In [ ]:
df = A.collect(OUT)
df.to_csv(f"{OUT}/summary.csv", index=False)
print(len(df), "runs;", f"{df.seconds.sum() / 3600:.1f} GPU-hours")